# Memory Optimization: Garbage Collection Tuning, Off-Heap, and Kryo

**Module**: 10_performance_and_optimization  
**Notebook**: 05_memory_optimization.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Techniques for memory optimization: JVM Garbage Collection diagnostics (G1GC), Kryo serialization, and off-heap memory.
- **Why does this topic exist?**: Memory leaks and GC pauses ruin job predictability and cause heartbeat loss.
- **What problem does it solve?**: Stop-The-World GC pauses exceeding 30 seconds crashing worker heartbeats.
- **Where is it used in real-world Data Engineering?**: Enterprise memory tuning on large JVM clusters.

---

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Memory_Opt_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
# Inspect serialization configuration
print("Serializer: ", spark.conf.get("spark.serializer"))
print("Is Kryo?:   ", "Kryo" in spark.conf.get("spark.serializer", ""))

---

## Experiment: Kryo Serialization Verification

- **1. What are we testing?**: Verify Kryo is active in our lab environment.
- **2. What do we expect?**: spark.serializer is org.apache.spark.serializer.KryoSerializer.

In [ ]:
print('Active serializer:', spark.sparkContext.getConf().get('spark.serializer'))

**3. What happened?**: KryoSerializer confirmed active.  
**4. Why did it happen?**: Configured in conf/spark-defaults.conf.  
**5. Practical Engineering Lesson**: Kryo is up to 10x faster and significantly more compact than standard Java serialization.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Off-Heap Setting
**Problem Statement**: Print spark.memory.offHeap.enabled.  
**Input / Context**: spark.conf

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): GC Pause Detection
**Problem Statement**: Where to look in UI for GC time.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): G1GC Flags
**Problem Statement**: List recommended JVM flags for Java 21.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Off-Heap Setting):
```python
print(spark.conf.get('spark.memory.offHeap.enabled','false'))
```

#### Solution 2 (GC Pause Detection):
```python
print('Check Task GC Time column in Stages tab; red bars in event timeline.')
```

#### Solution 3 (G1GC Flags):
```python
print('-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Kryo Serializer**: Always enable Kryo for faster network and storage serialization.
- **GC Tuning**: Use G1GC to keep pauses below 1 second.

### Essential Technical Terms
- `KryoSerializer`: High performance binary serializer.
- `Stop-The-World`: JVM GC pause freezing all application threads.

### Common Engineering Mistakes
- **Using default Java serialization**: Bulky serialized bytes, slow network transfer.

### Interview Questions on This Topic
- **Q: Why is KryoSerializer preferred over standard Java serialization?**
  - *A*: Kryo produces much smaller binary payloads and serializes objects significantly faster, reducing shuffle write time and network transfer volume.

### What to Study Next
Open 06_data_skew_optimization.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')